### PART - A

## Import and load dataset

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

In [2]:
dataset = sns.load_dataset('titanic')

##  1 - Series vs DataFrame

In [3]:
age = dataset['age']            # Series
age_frame = dataset[['age']]    # DataFrame

print(type(age))
print(type(age_frame))
print(age.shape, age_frame.shape)

# note to self : single brackets give a Series (1D , with an index and a name) , double brackets give a DataFrame (2D) even for one column

<class 'pandas.Series'>
<class 'pandas.DataFrame'>
(891,) (891, 1)


##  2 - loc and iloc

In [4]:
age_sorted = dataset['age'].dropna().sort_values()

print(age_sorted.loc[0])    # label 0
print(age_sorted.iloc[0])   # position 0

# note to self : after sort_values the index is shuffled , so loc[0] and iloc[0] are not the same row , loc works on the label and iloc on the position

22.0
0.42


##  3 - Vectorised operations

In [5]:
fare_hundreds = dataset['fare'] / 100
age_rounded = dataset['age'].round()
family = dataset['sibsp'] + dataset['parch']

print(fare_hundreds.head())
print(age_rounded.head())
print(family.head())

# note to self : no loop needed , the operation is applied to the whole Series at once

0    0.072500
1    0.712833
2    0.079250
3    0.531000
4    0.080500
Name: fare, dtype: float64
0    22.0
1    38.0
2    26.0
3    35.0
4    35.0
Name: age, dtype: float64
0    1
1    1
2    0
3    1
4    0
dtype: int64


##  4 - Exploring a Series

In [6]:
dataset['embarked'].unique()

<StringArray>
['S', 'C', 'Q', nan]
Length: 4, dtype: str

In [7]:
dataset['embarked'].nunique()

# note to self : unique keeps the NaN inside the array , nunique only counts the distinct non-null values , so 3 and not 4

3

In [8]:
print(dataset['embarked'].value_counts())
print(dataset['embarked'].value_counts(normalize=True))

# note to self : value_counts ignores the missing values by default , normalize=True gives the proportions instead of the counts

embarked
S    644
C    168
Q     77
Name: count, dtype: int64
embarked
S    0.724409
C    0.188976
Q    0.086614
Name: proportion, dtype: float64


In [9]:
print(dataset['fare'].min())
print(dataset['fare'].max())
print(dataset['fare'].idxmax())

# note to self : idxmax gives the index label of the maximum , not the maximum itself

0.0
512.3292
258


In [10]:
dataset['fare'].sort_values().head(10)

# note to self : sort_values keeps the original index , so the labels are shuffled , i only show the 10 cheapest here

806    0.0
413    0.0
481    0.0
466    0.0
179    0.0
263    0.0
633    0.0
674    0.0
271    0.0
277    0.0
Name: fare, dtype: float64

In [11]:
dataset['fare'].nlargest(5)

258    512.3292
679    512.3292
737    512.3292
27     263.0000
88     263.0000
Name: fare, dtype: float64

##  5 - Missing values

In [12]:
print(age.isna().sum())
print(age.notna().sum())
print(len(age))

# note to self : 177 missing ages out of 891

177
714
891


In [13]:
age_filled = age.fillna(age.mean())
print(age_filled.isna().sum())
print(age_filled.mean())

age_dropped = age.dropna()
print(len(age_dropped))
print(age_dropped.mean())

# note to self : fillna replaces the missing values , dropna removes the whole row , here the mean is the same before and after because i filled with the mean

0
29.69911764705882
714
29.69911764705882


##  6 - Transformation

In [14]:
print(dataset['pclass'].astype('int8').head())
print(dataset['age'].round().astype('Int64').head())

# note to self : astype changes the dtype , age is float64 because of the NaN , 'Int64' (capital I) is the nullable integer type
# note to self : the raw age has .5 values so astype('Int64') raises "cannot safely cast" , i have to round first

0    3
1    1
2    3
3    1
4    3
Name: pclass, dtype: int8
0    22
1    38
2    26
3    35
4    35
Name: age, dtype: Int64


In [15]:
sex_encoded = dataset['sex'].map({'male': 0, 'female': 1})
sex_encoded.value_counts()

# note to self : map replaces each value using the dictionary , any value missing from the dict becomes NaN

sex
0    577
1    314
Name: count, dtype: int64

In [16]:
def fare_band(f):
    if f < 10:
        return 'cheap'
    if f < 50:
        return 'medium'
    return 'expensive'

dataset['fare'].apply(fare_band).value_counts()

fare
medium       394
cheap        336
expensive    161
Name: count, dtype: int64

In [17]:
dataset['age'].apply(lambda x: 'child' if x < 12 else 'adult').value_counts()

# note to self : NaN < 12 is False so the missing ages end up as adult , np.where has the same trap , i deal with it in part B

age
adult    823
child     68
Name: count, dtype: int64

### PART - B

##  1 - Boolean mask

In [18]:
mask = dataset['age'] > 30

print(mask.head())
print(mask.sum())
print(mask.mean())

# note to self : a comparison on a Series returns a boolean Series , sum counts the True , mean gives the proportion , and NaN compared to anything is always False

0    False
1     True
2    False
3     True
4     True
Name: age, dtype: bool
305
0.34231200897867564


##  2 - Combining conditions

In [19]:
both = (dataset['age'] > 30) & (dataset['fare'] > 50)
either = (dataset['age'] > 30) | (dataset['age'] < 12)
not_old = ~(dataset['age'] > 30)

print(both.sum())
print(either.sum())
print(not_old.sum())

# note to self : the parentheses are mandatory , & has a higher precedence than > so without them python compares 30 & dataset['fare'] first and raises a TypeError
# note to self : and / or do not work on Series , they try to reduce the whole Series to one bool and raise "truth value is ambiguous" , so we use & | ~

83
373
586


In [20]:
# dataset['age'] > 30 & dataset['fare'] > 50   # TypeError , & binds tighter than >
# (dataset['age'] > 30) and (dataset['fare'] > 50)   # ValueError , truth value is ambiguous

# note to self : kept as comments so the notebook still runs , but these are the two exact errors to remember

##  3 - isin , between and isna

In [21]:
print(dataset['embarked'].isin(['C', 'Q']).sum())
print(dataset['fare'].between(10, 50).sum())
print(dataset['age'].isna().sum())
print(dataset['age'].notna().sum())

# note to self : isin for a list of values , between for an interval (both bounds included) , isna / notna for the missing values

245
395
177
714


##  4 - Three ways to filter

In [22]:
mask = dataset['age'] > 30

print(dataset[mask].shape)
print(dataset.loc[mask, ['age', 'fare', 'survived']].shape)
print(dataset.query('age > 30')[['age', 'fare', 'survived']].shape)

dataset.loc[mask, ['age', 'fare', 'survived']].head()

# note to self : the three give the same rows , loc adds the column choice , query is the readable string version , inside query a python variable needs @

(305, 15)
(305, 3)
(305, 3)


,age,fare,survived
1,38.0,71.2833,1
3,35.0,53.1000,1
4,35.0,8.0500,0
6,54.0,51.8625,0
11,58.0,26.5500,1


##  5 - Count and rate

In [23]:
survived = dataset['survived'] == 1

print(survived.sum())
print(survived.mean())
print(dataset['survived'].mean())

# note to self : sum of a mask = number of rows , mean = proportion , so the global survival rate is 0.3838 , and survived is already 0 / 1 so its mean is the same thing

342
0.3838383838383838
0.3838383838383838


##  6 - np.where and np.select

In [24]:
age = dataset['age']

dataset['is_child'] = np.where(age < 12, 'child', 'adult')
dataset['is_child'].value_counts()

# note to self : np.where(condition , value if True , value if False) is the vectorised if / else

is_child
adult    823
child     68
Name: count, dtype: int64

In [25]:
conditions = [age < 12, age < 18, age < 60]
choices = ['child', 'teen', 'adult']

age_group = np.select(conditions, choices, default='senior')
pd.Series(age_group).value_counts()

# note to self : np.select takes the first condition that is True , but NaN falls through to the default , so the 177 missing ages became senior

adult     575
senior    203
child      68
teen       45
Name: count, dtype: int64

In [26]:
age_group = np.where(age.isna(), 'unknown', age_group)
pd.Series(age_group).value_counts()

# note to self : always handle the NaN before np.select , otherwise the default catches them

adult      575
unknown    177
child       68
teen        45
senior      26
Name: count, dtype: int64

##  7 - Answering with logic

In [27]:
women_first = dataset[(dataset['sex'] == 'female') & (dataset['pclass'] == 1)]

print(len(women_first))
print(women_first['survived'].mean())

# note to self : 94 women in 1st class , 96.8% of them survived

94
0.9680851063829787


In [28]:
men_60 = dataset[(dataset['sex'] == 'male') & (dataset['age'] > 60)]

len(men_60)

# note to self : 19 men older than 60 , the 177 missing ages are silently excluded because NaN > 60 is False

19

In [29]:
solo_third = dataset[(dataset['sibsp'] == 0) & (dataset['parch'] == 0) & (dataset['pclass'] == 3)]

print(len(solo_third))
print(solo_third['survived'].mean())

# note to self : 324 passengers travelling alone in 3rd class , only 21.3% survived

324
0.21296296296296297


### PART - C

In [30]:
dataset = sns.load_dataset('titanic')

# note to self : i reload the raw data because part B added the is_child column , i want to inspect the original 15 columns

##  1 - Inspecting the dataset

In [31]:
print(dataset.shape)
dataset.info()

# note to self : info gives the dtype and the non-null count of every column , shape gives the (rows , columns)

(891, 15)
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    str     
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    str     
 8   class        891 non-null    category
 9   who          891 non-null    str     
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    str     
 13  alive        891 non-null    str     
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), str(5)
memory usage: 80.7 KB


In [32]:
dataset.describe(include='all')

# note to self : include='all' adds the categorical columns (count , unique , top , freq) , without it describe only looks at the numbers

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
count,891.000000,891.000000,891,714.000000,891.000000,891.000000,891.000000,889,891,891,891,203,889,891,891
unique,NaN,NaN,2,NaN,NaN,NaN,NaN,3,3,3,2,7,3,2,2
top,NaN,NaN,male,NaN,NaN,NaN,NaN,S,Third,man,True,C,Southampton,no,True
freq,NaN,NaN,577,NaN,NaN,NaN,NaN,644,491,537,537,59,644,549,537
mean,0.383838,2.308642,NaN,29.699118,0.523008,0.381594,32.204208,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,0.486592,0.836071,NaN,14.526497,1.102743,0.806057,49.693429,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,0.000000,1.000000,NaN,0.420000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,0.000000,2.000000,NaN,20.125000,0.000000,0.000000,7.910400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,0.000000,3.000000,NaN,28.000000,0.000000,0.000000,14.454200,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,1.000000,3.000000,NaN,38.000000,1.000000,0.000000,31.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [33]:
(dataset.isna().mean() * 100).round(2).sort_values(ascending=False)

# note to self : deck is missing 77.2% of the time , age 19.9% , embarked and embark_town 0.22% , the rest is complete

deck           77.22
age            19.87
embarked        0.22
embark_town     0.22
sex             0.00
pclass          0.00
survived        0.00
fare            0.00
parch           0.00
sibsp           0.00
class           0.00
adult_male      0.00
who             0.00
alive           0.00
alone           0.00
dtype: float64

##  2 - Numeric , nominal and ordinal columns

In [34]:
numeric_cols = dataset.select_dtypes(include='number').columns.tolist()
nominal_cols = dataset.select_dtypes(include=['str', 'category']).columns.tolist()
ordinal_cols = ['pclass']

print(numeric_cols)
print(nominal_cols)
print(ordinal_cols)

# note to self : pclass is int64 so it lands in numeric , but it is ordinal , 1st > 2nd > 3rd is a ranking and not a real number
# note to self : class and deck are 'category' dtype in pandas 3 , so i have to select str and category for the nominal ones
# note to self : class is pclass written as text , alive is survived as text , who is sex + child , so most of the nominal columns are duplicates , i keep them in mind before encoding anything

['survived', 'pclass', 'age', 'sibsp', 'parch', 'fare']
['sex', 'embarked', 'class', 'who', 'deck', 'embark_town', 'alive']
['pclass']


##  3 - groupby

In [35]:
dataset.groupby('sex')['survived'].agg(['mean', 'count'])

# note to self : mean is the survival rate , count the number of passengers , 74% of the women survived against 19% of the men

,mean,count
sex,,
female,0.742038,314
male,0.188908,577


In [36]:
dataset.groupby('pclass')['survived'].agg(['mean', 'count'])

# note to self : 63% in 1st , 47% in 2nd , 24% in 3rd , the class has a clear effect

,mean,count
pclass,,
1,0.629630,216
2,0.472826,184
3,0.242363,491


In [37]:
dataset.groupby(['sex', 'pclass'])['survived'].agg(['mean', 'count'])

# note to self : the effect of the class is still there inside each sex , 97% of the women in 1st against 50% in 3rd

mean  count
sex    pclass                 
female 1       0.968085     94
       2       0.921053     76
       3       0.500000    144
male   1       0.368852    122
       2       0.157407    108
       3       0.135447    347

##  4 - Age bands with pd.cut

In [38]:
age_bands = pd.cut(
    dataset['age'],
    bins=[0, 12, 18, 30, 60, 100],
    labels=['child', 'teen', 'young', 'adult', 'senior'],
)

dataset.groupby(age_bands, observed=True)['survived'].agg(['mean', 'count'])

# note to self : pd.cut cuts on the bin edges , pd.qcut cuts on the quantiles (same number of rows per bin)
# note to self : pd.cut is right-closed by default so a 12 year old goes into child (69 here , 68 in part B where i used < 12) , and NaN stays NaN so groupby drops those 177 rows

,mean,count
age,,
child,0.579710,69
teen,0.428571,70
young,0.355556,270
adult,0.420495,283
senior,0.227273,22


##  5 - Correlation

In [39]:
corr = dataset.corr(numeric_only=True)
corr.round(2)

# note to self : survived is the target , the diagonal is always 1

,survived,pclass,age,sibsp,parch,fare,adult_male,alone
survived,1.00,-0.34,-0.08,-0.04,0.08,0.26,-0.56,-0.20
pclass,-0.34,1.00,-0.37,0.08,0.02,-0.55,0.09,0.14
age,-0.08,-0.37,1.00,-0.31,-0.19,0.10,0.28,0.20
sibsp,-0.04,0.08,-0.31,1.00,0.41,0.16,-0.25,-0.58
parch,0.08,0.02,-0.19,0.41,1.00,0.22,-0.35,-0.58
fare,0.26,-0.55,0.10,0.16,0.22,1.00,-0.18,-0.27
adult_male,-0.56,0.09,0.28,-0.25,-0.35,-0.18,1.00,0.40
alone,-0.20,0.14,0.20,-0.58,-0.58,-0.27,0.40,1.00


In [40]:
corr['survived'].sort_values(ascending=False)

# note to self : adult_male (-0.56) and pclass (-0.34) are the strongest , but adult_male is just sex written as 0 / 1 (male) so it is redundant with sex
# note to self : the sign matters , pclass is negative because 1st class survives more , fare is positive because the rich cabins are on the upper decks
# note to self : this is linear correlation only , sex is not numeric so it never shows up here , that is why groupby is needed

survived      1.000000
fare          0.257307
parch         0.081629
sibsp        -0.035322
age          -0.077221
alone        -0.203367
pclass       -0.338481
adult_male   -0.557080
Name: survived, dtype: float64